In [1]:
import json
from typing import Literal
from pydantic import BaseModel, Field

from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_ollama import ChatOllama

import logging

import sys

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]  
)
logger = logging.getLogger(__name__)

In [3]:
SYSTEM_PROMPT = """You are an advanced triage and routing assistant for a specialized mental health support system. 
Your sole responsibility is to analyze the user's input and categorize it into exactly ONE of the allowed intent classes.

CRITICAL INSTRUCTIONS:
1. Base your classification entirely on the core intent of the user's statement.
2. Rely heavily on the few-shot examples provided below to understand the classification boundaries.
3. Choose exactly one of the allowed categories. Do not invent new categories.

ALLOWED CATEGORIES AND DEFINITIONS:

- greeting: 
  The user is starting a conversation, saying hello, or checking if someone is online.
  (e.g., "Hi", "Hello", "Is anyone there?", "Good morning")

- goodbye: 
  The user is ending the conversation, signing off, or indicating they are leaving.
  (e.g., "Bye", "See you later", "Talk to you tomorrow", "I'm heading out")

- gratitude: 
  The user is thanking the system, expressing appreciation, or confirming that their issue was resolved.
  (e.g., "Thank you so much", "Thanks for the help", "That makes sense, thank you", "Appreciate it")

- asking_mental_health_question: 
  The user is actively seeking help, coping mechanisms, definitions, or advice regarding mental health conditions, emotions, symptoms, or psychological well-being. This is a critical class that triggers our clinical knowledge retrieval pipeline.
  (e.g., "How do I deal with panic attacks?", "I'm feeling incredibly anxious right now", "What are the signs of burnout?", "Can you give me tips for depression?")

- out_of_scope: 
  The user is asking about general knowledge, coding, math, recipes, casual chit-chat, or anything completely unrelated to mental health support.
  (e.g., "What is the capital of France?", "Write a python script", "Tell me a joke", "How's the weather?")

Analyze the context carefully. If a user says "Hello, I am having a panic attack", the primary intent is 'asking_mental_health_question', not 'greeting'. Prioritize clinical inquiries over conversational fluff.
"""

In [4]:
class IntentResponse(BaseModel):
    intent: Literal[
        "greeting", 
        "goodbye", 
        "gratitude", 
        "asking_mental_health_question", 
        "out_of_scope"
    ] = Field(description="The classified intent of the user's message.")

In [5]:
class IntentClassifier:

    def __init__(
        self,
        model_name: str = "llama3.2",
        temperature: float = 0,
    ):
        logger.info(f"Initializing IntentClassifier with model='{model_name}', temp={temperature}")
        self.model_name = model_name
        self.temperature = temperature
        
        self.examples = self._load_examples()
        self.chain = self._build_chain()
        logger.info("IntentClassifier initialization complete.")

    def _load_examples(self):
        logger.info("Attempting to load few-shot examples from 'intentExamples.json'...")
        try:
            with open("intentExamples.json", "r", encoding="utf-8") as f:
                examples = json.load(f)
            logger.info(f"Successfully loaded {len(examples)} few-shot examples.")
            return examples
        except Exception as e:
            logger.error(f"Failed to load examples file: {str(e)}")
            raise

    def _build_chain(self):
        logger.info("Assembling LangChain LCEL pipeline components...")
        
        example_prompt = PromptTemplate(
            input_variables=["query", "intent"],
            template="User: {query}\nIntent: {intent}",
        )

        few_shot_prompt = FewShotPromptTemplate(
            examples=self.examples,
            example_prompt=example_prompt,
            prefix=SYSTEM_PROMPT,
            suffix="User: {input}\nIntent:",
            input_variables=["input"],
        )

        system_message_prompt = SystemMessagePromptTemplate(prompt=few_shot_prompt)
        human_message_prompt = HumanMessagePromptTemplate.from_template("{input}")

        chat_prompt = ChatPromptTemplate.from_messages(
            [system_message_prompt, human_message_prompt]
        )
        logger.info("Prompt templates structured successfully.")

        logger.info(f"Connecting to local Ollama instance for model '{self.model_name}'...")
        llm = ChatOllama(
            model=self.model_name,
            temperature=self.temperature,
        )

        logger.info("Binding Pydantic output schema (IntentResponse) to LLM...")
        structured_llm = llm.with_structured_output(IntentResponse)

        chain = chat_prompt | structured_llm
        logger.info("LCEL routing chain compiled successfully.")
        return chain

    def predict(self, text: str) -> str:
        logger.info(f"Received user input for prediction: '{text}'")
        logger.info("Invoking LLM chain (waiting for local Ollama response)...")
        
        try:
            prediction: IntentResponse = self.chain.invoke({"input": text})
            logger.info(f"LLM successfully responded. Extracted parsed intent: '{prediction.intent}'")
            return prediction.intent
        except Exception as e:
            logger.error(f"Error encountered during chain execution or Pydantic validation: {str(e)}")
            raise

    def save(self):
        import os
        logger.info("Saving configuration settings...")
        os.makedirs("saved", exist_ok=True)
        config = {
            "model_name": self.model_name,
            "temperature": self.temperature,
        }
        with open("saved/classifier_config.json", "w", encoding="utf-8") as f:
            json.dump(config, f, indent=4)
        logger.info("Configuration saved successfully to 'saved/intent_classifier_config.json'.")

    @classmethod
    def load(cls):
        logger.info("Loading configuration from file...")
        with open("saved/intent_classifier_config.json", "r", encoding="utf-8") as f:
            config = json.load(f)
        logger.info(f"Config loaded: {config}. Re-instantiating class...")
        return cls(**config)

In [7]:
def test_system():
    logger.info("Initializing classifier...")
    classifier = IntentClassifier(model_name="llama3.2", temperature=0)

    test_cases = {
        "Hey, how's it going?": "greeting",
        "Goodbye, see you tomorrow.": "goodbye",
        "Thank you so much for the advice!": "gratitude",
        "I'm feeling deeply anxious and can't sleep.": "asking_mental_health_question",
        "Can you write a poem about space?": "out_of_scope"
    }

    logger.info("="*20 + " Starting Tests " + "="*20)
    
    passed_counts = 0
    total_tests = len(test_cases)

    logging.getLogger("langchain").setLevel(logging.WARNING)

    for phrase, expected in test_cases.items():
       
        logger.info(f"Testing Input: \"{phrase}\"")
        try:
            result = classifier.predict(phrase)
            is_correct_type = isinstance(result, str)
            
            if result == expected and is_correct_type:
                status = "PASS"
                passed_counts += 1
            elif not is_correct_type:
                status = "FAIL (Returned object is not a primitive string type)"
            else:
                status = f"FAIL (Model predicted '{result}' instead of '{expected}')"
            
            logger.info(f"Result Status: {status}")
            logger.info("-" * 50)
            
        except Exception as e:
            logger.critical(f"CRITICAL ERROR executing this test case: {str(e)}")
            logger.critical("-" * 50)

    
    logger.info("="*20 + " TEST SUMMARY " + "="*20)
    logger.info(f"Total Test Cases Run: {total_tests}")
    logger.info(f"Total Passed: {passed_counts} / {total_tests}")
    
    if passed_counts == total_tests:
        logger.info("SUCCESS: Your structured Pydantic intent router is working flawlessly!")
    else:
        logger.critical("NOTICE: Some test cases failed. Check your few-shot examples or model configurations.")

test_system()

14:49:12 [INFO] Initializing classifier...


14:49:12 [INFO] Initializing IntentClassifier with model='llama3.2', temp=0
14:49:12 [INFO] Attempting to load few-shot examples from 'intentExamples.json'...
14:49:12 [INFO] Successfully loaded 11 few-shot examples.
14:49:12 [INFO] Assembling LangChain LCEL pipeline components...
14:49:12 [INFO] Prompt templates structured successfully.
14:49:12 [INFO] Connecting to local Ollama instance for model 'llama3.2'...
14:49:12 [INFO] Binding Pydantic output schema (IntentResponse) to LLM...
14:49:12 [INFO] LCEL routing chain compiled successfully.
14:49:12 [INFO] IntentClassifier initialization complete.
14:49:12 [INFO] ==================== Starting Tests ====================
14:49:12 [INFO] Testing Input: "Hey, how's it going?"
14:49:12 [INFO] Received user input for prediction: 'Hey, how's it going?'
14:49:12 [INFO] Invoking LLM chain (waiting for local Ollama response)...
14:49:36 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
14:49:54 [INFO] LLM successfully 

In [6]:
def route_request(intent: str):

    if intent == "asking_mental_health_question":

        return (
            "TRIGGER_RAG_PIPELINE"
        )

    elif intent in [
        "greeting",
        "goodbye",
        "gratitude",
    ]:

        return (
            "DIRECT_RESPONSE"
        )

    else:

        return (
            "OUT_OF_SCOPE_FALLBACK"
        )

In [7]:
def main():
    print("Initializing IntentClassifier...")
    classifier = IntentClassifier()
    print("Classifier initialized successfully!")

    print("Saving classifier...")
    classifier.save()
    print("Classifier saved successfully!")

    print("Entering loop... (Type 'exit' to quit)")
    while True:
        user_input = input("\nUser: ")
        if user_input.lower() == "exit":
            break

        intent = classifier.predict(user_input)
        print(f"\nPredicted Intent: {intent}")

        route = route_request(intent)
        print(f"Route: {route}")
main()

Initializing IntentClassifier...
15:15:15 [INFO] Initializing IntentClassifier with model='llama3.2', temp=0
15:15:15 [INFO] Attempting to load few-shot examples from 'intentExamples.json'...
15:15:15 [INFO] Successfully loaded 11 few-shot examples.
15:15:15 [INFO] Assembling LangChain LCEL pipeline components...
15:15:15 [INFO] Prompt templates structured successfully.
15:15:15 [INFO] Connecting to local Ollama instance for model 'llama3.2'...
15:15:15 [INFO] Binding Pydantic output schema (IntentResponse) to LLM...
15:15:15 [INFO] LCEL routing chain compiled successfully.
15:15:15 [INFO] IntentClassifier initialization complete.
Classifier initialized successfully!
Saving classifier...
15:15:15 [INFO] Saving configuration settings...
15:15:15 [INFO] Configuration saved successfully to 'saved/intent_classifier_config.json'.
Classifier saved successfully!
Entering loop... (Type 'exit' to quit)
15:15:15 [INFO] Received user input for prediction: ''
15:15:15 [INFO] Invoking LLM chain (wa